# Figure 2. Genetic architecture and cross-phenotype characterization of glycoproteomic QTLs (MAF >= 0.05)

In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(treemapify)
library(RColorBrewer)
library(tidyverse)
library(data.table)

options(repr.plot.width = 6, repr.plot.height = 8)

qtl_counts <- readRDS('data/Figure_2a.rds') %>%
  mutate(
    GlycanTypeSummary = case_when(
      GlycanTypeSummary == 'glycosylation' ~ 'Glycosylation',
      GlycanTypeSummary == 'glycation'     ~ 'Glycation',
      GlycanTypeSummary == 'mixed'         ~ 'Dual modification',
      TRUE ~ GlycanTypeSummary
    ),
    # # stack order: Glycosylation
    GlycanTypeSummary = factor(GlycanTypeSummary,
                               levels = c('Glycosylation', 'Glycation', 'Dual modification')),
    Adjustment = recode(Adjustment, 'Adjusted' = 'Protein-adjusted'),
    Adjustment = factor(Adjustment, levels = c('Unadjusted', 'Protein-adjusted'))
  )

glycan_colors <- c(
  'Glycosylation'     = '#4C72B0',
  'Glycation'         = '#C44E52',
  'Dual modification' = 'grey50'
)

f2a <- ggplot(qtl_counts, aes(x = Adjustment, y = NumGpID, fill = GlycanTypeSummary)) +
  geom_col(width = 0.7) +
  labs(
    x    = NULL,
    y    = 'Glycopeptiforms with a cis-glycoQTL',
    fill = NULL
  ) +
  scale_fill_manual(values = glycan_colors) +
  theme_classic(base_size = 16) +
  theme(
    legend.position = c(0.87, 0.9),
    axis.text.x     = element_text(angle = 45, hjust = 1, size = 16)
  ) +
  coord_cartesian(clip = "off")

## Figure 2b

In [ ]:
enrich_df <- readRDS('data/Figure_2b.rds') %>%
  mutate(
    GlycanTypeSummary = case_when(
      GlycanTypeSummary == 'glycosylation' ~ 'Glycosylation',
      GlycanTypeSummary == 'glycation'     ~ 'Glycation',
      GlycanTypeSummary == 'mixed'         ~ 'Dual modification',
      TRUE ~ GlycanTypeSummary
    ))
f2b <-  ggplot(enrich_df,
                   aes(x = reorder(GlycanTypeSummary, -odds_ratio),
                       y = odds_ratio,
                       fill = GlycanTypeSummary)) +
  geom_col(width = 0.7) +
  geom_errorbar(aes(ymin = ci_lower, ymax = ci_upper), width = 0.2, size = 0.7) +
  geom_hline(yintercept = 1, linetype = "dashed") +
  labs(x = "",
       y = "Odds ratio for harboring a cis-glycoQTL") +
  theme_classic(base_size = 16) +
  theme(
    legend.position = "none",
    axis.text.x = element_text(angle = 45, hjust = 1, size = 16)   # rotate x-axis labels
  ) +
  scale_fill_manual(values = c("Glycation" = "#C44E52", "Glycosylation" = "#4C72B0"))

options(repr.plot.width=3, repr.plot.height=7)

f2b
# ggsave('f2b.pdf')

## Figure 2C

In [ ]:
enrich_df <- readRDS('data//Figure_2c.rds') %>% dplyr::filter(GlycanClass!='Glycation')
# glycan-class palette shared across Fig 1c/2c/3
cus_colors <- c(
  "Paucimannose" = "#A6CEE3",
  "Hybrid/Complex" = "#1F78B4",
  "High-mannose" = "#66C2A5",
  "Glycation" = "#C44E52",
  "Other" = "grey50"
)
f2c <- ggplot(enrich_df,
                   aes(x = reorder(GlycanClass, -odds_ratio),
                       y = odds_ratio,
                       fill = GlycanClass)) +
  geom_col(width = 0.7) +
  geom_errorbar(aes(ymin = ci_lower, ymax = ci_upper), width = 0.2, size = 0.7) +
  geom_hline(yintercept = 1, linetype = "dashed") +
  labs(x = "",
       y = "Odds ratio for harboring a cis-glycoQTL") +
  theme_classic(base_size = 14) +
  theme(
    legend.position = "none",
    axis.text.x = element_text(angle = 45, hjust = 1, size = 16),
    axis.text.y = element_text(size = 12.8),  # match panel b's y-axis number size (theme_classic(base_size=16) default)
    axis.title.y = element_text(size = 16)
  ) +
  scale_fill_manual(values = cus_colors) +
  scale_x_discrete(labels = c("Hybrid/Complex" = "Hybrid/complex"))

## Figure 2d

In [ ]:
library(dplyr)
library(stringr)
library(ggplot2)
library(tidyr)
plot_data_tmp <- readRDS('data//Figure_2d.rds')  %>% dplyr::filter(study == 'glycosylation_unadjusted_maf05')
x_limits <- list(
  min_val = min(plot_data_tmp$Enrichment_log2 - plot_data_tmp$Enrichment_SE_log2, na.rm = TRUE),
  max_val = max(plot_data_tmp$Enrichment_log2 + plot_data_tmp$Enrichment_SE_log2, na.rm = TRUE)
)

function_order <- c("ATAC", "H3K27ac", "H3K4me3", "Super-enhancer")
celltype_order <- c("Astrocyte", "Neuron", "Oligodendrocyte", "Microglia")

plot_data_tmp <- plot_data_tmp %>%
  mutate(
    Function = factor(Function, levels = function_order),
    CellType = factor(CellType, levels = celltype_order)
  )

# background shading per cell-type block, one rect per Function row within each CellType
anno_bg <- plot_data_tmp %>%
  distinct(Function, CellType) %>%
  mutate(y = as.numeric(Function), ymin = y - 0.5, ymax = y + 0.5, xmin = x_limits$min_val, xmax = x_limits$max_val)

anno_bg$bg_fill <- c(Astrocyte = "#FCE8E6", Neuron = "#EAF2FB", Oligodendrocyte = "#EAF6EA", Microglia = "#FFF3E0")[as.character(anno_bg$CellType)]

f2d <- ggplot(plot_data_tmp, aes(Enrichment_log2, Function)) +
  geom_rect(data = anno_bg, aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax), inherit.aes = FALSE, fill = anno_bg$bg_fill, alpha = 0.32) +
  geom_vline(xintercept = 0, linetype = "dashed", linewidth = 0.45, color = "grey45") +
  geom_col(aes(fill = Function), width = 0.62, color = "grey18", linewidth = 0.22) +
  geom_errorbar(aes(xmin = Enrichment_log2 - Enrichment_SE_log2, xmax = Enrichment_log2 + Enrichment_SE_log2), width = 0.18, linewidth = 0.4, color = "grey18") +
  facet_grid(CellType ~ ., scales = "free_y", space = "free_y", switch = "y") +
  coord_cartesian(clip = "off") +
  scale_x_continuous(expand = expansion(mult = c(0.05, 0.02)))+
  scale_fill_manual(values = c(ATAC = "#4E79A7", H3K27ac = "#E15759", H3K4me3 = "#76B7B2", `Super-enhancer` = "#B07AA1")) +
  theme_classic(base_size = 16) +
  theme(strip.background = element_rect(fill = "grey97", color = "grey75", linewidth = 0.35),
        strip.text.y.left = element_text(face = "bold", angle = 90, size = 11),
        strip.placement = "outside",
        legend.position = "none",
        panel.grid.major.x = element_line(color = "grey90", linewidth = 0.35),
        panel.grid.minor = element_blank(), panel.grid.major.y = element_blank(), axis.line.y = element_blank(),
        axis.ticks.y = element_blank(),
        axis.text.y = element_text(size = 14)) +
  labs(title = NULL, x = expression(log[2]("Enrichment score")), y = NULL, fill = NULL)

options(repr.plot.width=8, repr.plot.height=8)

f2d
# ggsave('f2d.pdf')

## Figure 2E

In [ ]:
coloc_df <- readRDS('data//Figure_2e.rds')
# Build with the ORIGINAL short codes
contexts_ordered <- names(sort(colSums(coloc_df), decreasing = TRUE))

max_size <- max(colSums(coloc_df))

f2e <- UpSetR::upset(coloc_df, 
      order.by = "freq", 
      keep.order = TRUE,
      sets = contexts_ordered,  # ordered sets
      nsets = length(contexts_ordered),
      main.bar.color = "steelblue3", 
      sets.bar.color = "sandybrown", 
      text.scale = c(2,2,2,2,2),
      matrix.color = "black",
      number.angles = 30,
      mb.ratio = c(0.5, 0.5),
      point.size = 4, 
      line.size = 1.5,
      set_size.show = TRUE,
      set_size.angles = 0,
      set_size.numbers_size = 7,
      set_size.scale_max = max_size + 0.25*max_size,
      nintersects = 100,
      mainbar.y.label = "Shared QTLs",
      sets.x.label = NULL)

set_labels <- c(Glyco = "Glycopeptiforms", Ast = "Astrocytes", Mic = "Microglia",
                 Exc = "Excitatory", Oli = "Oligodendrocytes", OPC = "OPC", Inh = "Inhibitory")
relabel_grob <- function(g, mapping) {
  if (is.null(g)) return(g)
  lab <- tryCatch(g$label, error = function(e) NULL)
  if (!is.null(lab) && all(trimws(lab) %in% names(mapping))) {
    g$label <- mapping[trimws(lab)]
    if (!is.null(g$gp)) g$gp$fontsize <- 13
  }
  if (!is.null(g$children)) {
    for (nm in names(g$children)) g$children[[nm]] <- relabel_grob(g$children[[nm]], mapping)
  }
  if (inherits(g, "gtable")) {
    for (i in seq_along(g$grobs)) g$grobs[[i]] <- relabel_grob(g$grobs[[i]], mapping)
  }
  g
}
f2e$Matrix <- relabel_grob(f2e$Matrix, set_labels)

axisl_col <- unique(f2e$Matrix$layout$l[f2e$Matrix$layout$name == "axis-l"])
f2e$Matrix$widths[axisl_col] <- unit(0.4, "null") + unit(3.2, "cm") + unit(0.6, "null")

tighten_set_size_counts <- function(g, target) {
  lab <- tryCatch(g$label, error = function(e) NULL)
  if (!is.null(lab) && identical(as.character(lab), target)) {
    g$hjust <- rep(1, length(lab))
  }
  if (!is.null(g$children)) {
    for (nm in names(g$children)) g$children[[nm]] <- tighten_set_size_counts(g$children[[nm]], target)
  }
  if (inherits(g, "gtable")) {
    for (i in seq_along(g$grobs)) g$grobs[[i]] <- tighten_set_size_counts(g$grobs[[i]], target)
  }
  g
}
set_size_counts <- as.character(sort(colSums(coloc_df), decreasing = TRUE))
f2e$Sizes <- tighten_set_size_counts(f2e$Sizes, set_size_counts)
f2e$Sizes$widths[length(f2e$Sizes$widths)] <- unit(0.2, "lines")

sizes_axis_b_row <- unique(f2e$Sizes$layout$t[f2e$Sizes$layout$name == "axis-b"])
matrix_axis_b_row <- unique(f2e$Matrix$layout$t[f2e$Matrix$layout$name == "axis-b"])
f2e$Matrix$heights[matrix_axis_b_row] <- f2e$Sizes$heights[sizes_axis_b_row]

## Figure 2F

In [ ]:
# Panel F now shows the standardized S-LDSC coefficient tau*
tau_list <- readRDS('data/Figure_2f_taustar.rds')
traits_dt_hans <- tau_list$data %>% dplyr::filter(trait != "Alzheimer's Disease (Jansen 2019)")
traits_dt_hans$trait_in_panel <- forcats::fct_drop(traits_dt_hans$trait_in_panel)

trait_relabel <- c(
  "Frontotemporal Dementia (Meta-analysis)"        = "Frontotemporal dementia (Meta-analysis)",
  "Alzheimer's Disease (Bellenguez)"                = "Alzheimer's disease (Bellenguez)",
  "Parkinson's Disease (Nalls 2019 excl. 23andMe)"  = "Parkinson's disease (Nalls 2019 excl. 23andMe)",
  "Alzheimer's Disease (Wightman Full)"             = "Alzheimer's disease (Wightman full)",
  "Alzheimer's Disease (Bellenguez EADB)"           = "Alzheimer's disease (Bellenguez EADB)",
  "Alzheimer's Disease (Kunkle Stage 1)"            = "Alzheimer's disease (Kunkle stage 1)",
  "Multiple Sclerosis"                              = "Multiple sclerosis",
  "Anorexia Nervosa"                                = "Anorexia nervosa",
  "Autism Spectrum Disorder"                        = "Autism spectrum disorder",
  "Cognitive Performance"                           = "Cognitive performance",
  "Reaction Time (Davies 2018)"                     = "Reaction time (Davies 2018)",
  "Amygdala Volume"                                 = "Amygdala volume",
  "Rostral Middle Frontal Thickness"                = "Rostral middle frontal thickness",
  "Caudate Volume"                                  = "Caudate volume",
  "Putamen Volume"                                  = "Putamen volume",
  "Brainstem Volume"                                = "Brainstem volume",
  "Superior Frontal Thickness"                      = "Superior frontal thickness"
)

category_relabel <- c(
  "Neurodegenerative /\n Neurological Disorders" = "Neurodegenerative /\n Neurological disorders",
  "Psychiatric /\n Cognitive Traits"              = "Psychiatric /\n Cognitive traits",
  "Brain Imaging /\n Morphometry"                 = "Brain imaging /\n Morphometric\n traits"
)

relabel_prefix <- function(x, map) {
  vapply(x, function(s) {
    parts  <- strsplit(s, "___", fixed = TRUE)[[1]]
    prefix <- if (parts[1] %in% names(map)) map[[parts[1]]] else parts[1]
    if (length(parts) > 1) paste0(prefix, "___", parts[2]) else prefix
  }, character(1), USE.NAMES = FALSE)
}

traits_dt_hans <- traits_dt_hans %>%
  mutate(
    trait          = dplyr::recode(trait, !!!trait_relabel),
    category       = factor(category_relabel[as.character(category)],
                             levels = category_relabel[levels(category)]),
    trait_in_panel = factor(relabel_prefix(as.character(trait_in_panel), trait_relabel),
                             levels = relabel_prefix(levels(trait_in_panel), trait_relabel))
  )

In [ ]:
library(dplyr); library(ggplot2); library(forcats); library(tidytext); library(ggrepel)

f2f <- ggplot(traits_dt_hans, aes(x = tau_star, y = trait_in_panel, color = color)) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "grey50") +
  geom_point(size = 3) +
  geom_errorbarh(aes(xmin = taus_lo, xmax = taus_hi), height = 0.3) +
  geom_text(
    data = subset(traits_dt_hans, sig),
    aes(x = taus_hi,
        label = case_when(coef_p2 < 0.001 ~ "***", coef_p2 < 0.01 ~ "**", coef_p2 < 0.05 ~ "*", TRUE ~ "")),
    size = 7, hjust = -0.2, vjust = 0.75, show.legend = FALSE
  ) +
  scale_y_reordered() +
  scale_color_identity() +
  facet_grid(category ~ ., scales = "free_y", space = "free_y") +
  labs(x = expression("Standardized effect size "*tau*"* (95% CI)"), y = NULL) +
  theme_minimal(base_size = 14) +
  theme(
    strip.text = element_text(face = "bold", size = 11),
    axis.text.y = element_text(size = 14),
    axis.text.x = element_text(size = 14),
    axis.title = element_text(size = 14),
    axis.title.x = element_text(size = 18),
    panel.grid.minor = element_blank(),
    panel.grid.major.y = element_blank(),
    strip.placement = "outside"
  )

options(repr.plot.width=10, repr.plot.height=8)
f2f
# ggsave( 'f2f.pdf')

## Combine

In [ ]:
# Make all plots share the same *final grob height*
library(ggplot2)
library(patchwork)
library(cowplot)
library(grid)

bm <- 20

f2a <- f2a + theme(plot.margin = margin(t = 20, r = 20, b = 30, l = 5))
f2a_legend <- cowplot::get_legend(
  f2a + theme(
    legend.position = "right",
    legend.key.width = unit(0.65, "lines"),
    legend.key.height = unit(0.5, "lines"),
    legend.spacing.x = unit(0.05, "lines"),
    legend.text = element_text(size = 13, margin = margin(l = 1)),
    legend.margin = margin(r = -10)
  )
)
f2a <- f2a + theme(legend.position = "none")
f2b <- f2b + theme(plot.margin = margin(t = 20, r = 0, b = 30, l = 40))
f2c <- f2c + theme(plot.margin = margin(t = 20, r = 0, b = 20, l = 40))

f2e_left <- cowplot::plot_grid(NULL, f2e$Sizes,
                              ncol = 1, rel_heights = c(1.5, 1))
f2e_right <- cowplot::plot_grid(f2e$Main_bar, f2e$Matrix,
                               ncol = 1, align = "v", axis = "lr",
                               rel_heights = c(1.5, 1))
f2e_wrapped <- cowplot::plot_grid(f2e_left, f2e_right,
                                  nrow = 1, rel_widths = c(0.9, 3)) +
  theme(plot.margin = margin(t = 0, r = 40, b = bm, l = 0))

H <- unit(14, "cm")

box <- function(p, h = H) ggdraw(p) + theme(plot.margin = margin(0,0,0,0)) +
  plot_grid(ggplotGrob(p), ncol = 1) # keep as grob

f2a_b <- ggdraw() + draw_plot(f2a, 0, 0, 1, 1) + theme(plot.margin = margin(0,0,0,0))
f2b_b <- ggdraw() + draw_plot(f2b, 0, 0, 1, 1) + theme(plot.margin = margin(0,0,0,0))
f2c_b <- ggdraw() + draw_plot(f2c, 0, 0, 1, 1) + theme(plot.margin = margin(0,0,0,0))
f2e_b <- ggdraw() + draw_plot(f2e_wrapped, 0, 0, 1, 1) + theme(plot.margin = margin(0,0,0,0))
 

options(repr.plot.width = 21, repr.plot.height = 14)

f2d <- f2d + theme(plot.margin = margin(t = 0, r = 40, b = 0, l = 40)) 
f2f <- f2f + theme(plot.margin = margin(t = 0, r = 55, b = 0, l = 40)) 
f2d_b <- ggdraw() + draw_plot(f2d, 0, 0, 1, 1) + theme(plot.margin = margin(0,0,0,0))
f2f_b <- ggdraw() + draw_plot(f2f, 0, 0, 1, 1) + theme(plot.margin = margin(0,0,0,0))
 

In [ ]:
library(patchwork)
library(ggplot2)

# ---- Top row: tags a-c, d (UpSet panel, swapped from "e" per manuscript order) ----
top_row_base <- (f2a_b + labs(tag = "a") |
            f2b_b + labs(tag = "b") |
            f2c_b + labs(tag = "c") |
            f2e_b + labs(tag = "e")) +
  plot_layout(widths = c(0.7, 0.7, 1.1, 3.2)) &
  theme(
    plot.tag.position = c(0, 1),
    plot.tag = element_text(
      face = "bold",
      size = 22,
      margin = margin(t = 4, r = 4, b = 4, l = 4)
    )
  )

top_row <- ggdraw() +
  draw_plot(top_row_base, 0, 0, 1, 1) +
  draw_grob(f2a_legend, x = 0.058, y = 0.825,
            width = 0.10, height = 0.14,
            clip = "off", halign = 0, valign = 1)

mid_row <- (f2d_b + labs(tag = "d") |
            f2f_b + labs(tag = "f"))+
  plot_layout(widths = c(0.7, 1)) &
  theme(
    plot.tag.position = c(0, 1),
    plot.tag = element_text(
      face = "bold",
      size = 22,
      margin = margin(t = 4, r = 4, b = 4, l = 4)
    )
  )

options(repr.plot.width = 21, repr.plot.height = 14)
combined_fig <- cowplot::plot_grid(
  top_row,
  mid_row,
  ncol = 1,
  rel_heights = c(1, 1.0),
  label_fontface = "bold"
)

ggsave("Figure2.pdf", combined_fig, width = 19, height = 13, dpi = 300)